# RAG Engine: Question Answering over the Knowledge Base

This notebook implements a retrieval-augmented generation (RAG) engine on top of the ChromaDB knowledge base
built in `01_knowledge_base_setup.ipynb`. It rewrites user questions into search queries, retrieves and reranks
relevant chunks, and generates answers grounded only in the retrieved context, with short conversation memory.

### Main goals
- rewrite user questions into self-contained search queries
- retrieve relevant chunks from ChromaDB using OpenAI embeddings
- rerank candidates with an LLM to keep only the most useful contexts
- generate grounded answers in the language of the user's question
- keep a short conversation history and export it to JSON and TXT

### Pipeline overview
| Step | Function | Purpose |
|---|---|---|
| 1 | `rewrite_query` | turn the question (and recent history) into search queries |
| 2 | `embedding_retrieval` | find the nearest chunks in ChromaDB |
| 3 | `rerank_contexts` | keep the most useful chunks |
| 4 | `generate_answer` | write the final answer from the contexts |

---
## Imports and Configuration

In [1]:
import os
import re
import json
from pathlib import Path
from datetime import datetime
from config.settings import DATA_DIR, RESULTS_TAB

from dotenv import load_dotenv
from openai import OpenAI
from chromadb import PersistentClient

# Load environment variables
load_dotenv(override=True)

# Check that the API key exists
api_key = os.getenv('OPENROUTER_API_KEY')
if not api_key:
    raise ValueError('Missing OPENROUTER_API_KEY')

# OpenRouter client (OpenAI-compatible API)
client = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)

# Model selection
EMBEDDING_MODEL = 'openai/text-embedding-3-small' # 8k context; .2/M input
CHAT_MODEL = 'anthropic/claude-sonnet-5.5'        # 1M context; 2/M input; 10/M output

# Processing parameters
EMBEDDING_MODEL = 'openai/text-embedding-3-small'
CHAT_MODEL = 'anthropic/claude-sonnet-5.5'
RETRIEVAL_K = 20       # chunks retrieved per query
RERANKED_K = 6         # chunks kept after reranking
HISTORY_MESSAGES = 6   # last 6 messages = 3 question/answer pairs sent to the LLM

data_dir = Path(DATA_DIR)
results_tab = Path(RESULTS_TAB)

---
## Load ChromaDB Knowledge Base

In [2]:
# Connect to the persistent ChromaDB created in 01_knowledge_base_setup.ipynb
chroma = PersistentClient(path=str(data_dir / 'chroma_db'))
collection = chroma.get_collection(name='knowledge_base')

assert collection.count() > 0, 'ChromaDB collection is empty' # Fail early if the knowledge base is empty

print("ChromaDB collection size:", collection.count())

ChromaDB collection size: 436


---
## Query Rewriting

The knowledge base is in English, so every question is rewritten into short English search queries.
Recent conversation history is used only to resolve references such as "it" or "that",
so follow-up questions become fully self-contained. Multi-part questions are split into separate queries.

In [3]:
REWRITE_PROMPT = '''
Rewrite the user question into short English search queries for a knowledge base
of project READMEs, notebooks, code and figures.

Goals:
- Preserve the original intent
- The knowledge base is in English, so write the queries in English regardless of the question language
- Use the conversation history only to resolve references like "it", "that", "the previous one", so every query is fully self-contained
- If the question is unrelated to the history, ignore the history
- Expand vague wording into retrieval-friendly terminology
- Split multi-part questions into separate queries, one per line
- Keep project names, numbers and technical terms exactly (e.g. lift, anomaly, DAX)
- Add likely technical synonyms (e.g. "computed" -> feature, formula, score, method) only when useful
- If user requests scoring, keep the requested scale
- If the user specifies a persona or expertise level, preserve it
- If the user asks to evaluate a project, include likely evidence dimensions:
  project goal, business value, data, methodology, experiments, reproducibility, strengths, weaknesses, limitations, and overall verdict

Return only the queries, one per line.
'''

def rewrite_query(question, conversation=None):
    history_text = ''
    if conversation:
        turns = conversation[-(HISTORY_MESSAGES // 2):]
        history_text = 'Conversation history:\n' + '\n'.join(
            f"user: {t['question']}\nassistant: {t['answer'][:500]}" for t in turns
        ) + '\n\n'

    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {'role': 'system', 'content': REWRITE_PROMPT},
                {'role': 'user', 'content': f'{history_text}Question:\n{question}'}
            ],
            temperature=0.1,   # near-deterministic rewriting
            max_tokens=500
        )
        
        # One query per line; strip bullet characters the model may add
        lines = (resp.choices[0].message.content or '').splitlines()
        queries = [q.strip('-• ').strip() for q in lines if q.strip()]
        return queries or [question]
    except Exception as e:
        print(f'rewrite_query failed: {e}')
        return [question]

---
## Retrieval

All queries are embedded in a single API call and searched in ChromaDB in a single request.
Results from different queries are mixed by rank, so every query contributes its best chunks first.
Duplicate chunks are removed.

In [4]:
# CELL 4: Retrieval
def embedding_retrieval(queries, k=RETRIEVAL_K):

    # One API call for all queries
    emb = client.embeddings.create(model=EMBEDDING_MODEL, input=queries)
    embeddings = [d.embedding for d in emb.data]

    # One ChromaDB request for all vectors
    res = collection.query(
        query_embeddings=embeddings,
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    # Separate list of chunks for each query (best first); score = 1 - distance
    per_query = [
        [{
            "text": doc,
            "project": meta.get("project", ""),
            "source": meta.get("source", ""),
            "headline": meta.get("headline", ""),
            "score": 1 - dist
            }
            for doc, meta, dist in zip(docs, metas, dists)
        ]
        for docs, metas, dists in zip(
            res["documents"], res["metadatas"], res["distances"]
        )
    ]

    # Merge in rank order (1st of every query, then 2nd, ...) and skip duplicate texts
    seen, results = set(), []
    for rank in range(k):
        for chunks in per_query:
            if rank < len(chunks) and chunks[rank]["text"] not in seen:
                seen.add(chunks[rank]["text"])
                results.append(chunks[rank])
                if len(results) == k:
                    return results
    return results

---
## Reranking

Vector search is fast but approximate. An LLM reads the candidates (first 600 characters of each)
and selects the most useful ones, favoring concrete definitions, methods, formulas and results,
while covering every part of the question.

In [5]:
# CELL 5: Rerank context
RERANK_PROMPT = '''You are a relevance reranker.

Select the {top_k} candidates most useful for answering the question, best first.
- Cover EVERY part of the question (e.g. both "how was X computed" and "what is Y")
- Prefer concrete definitions, formulas, methods and results over charts and overviews
- Use the 1-based numbers shown in brackets

Return only a JSON list of numbers, e.g. [3, 1, 7]'''

def rerank_contexts(question, candidates, top_k=RERANKED_K):
    if not candidates:
        return []

    # Number the candidates so the LLM can answer with indices only
    packed = '\n\n'.join(
        f"[{i}] SOURCE: {c['source']}\nHEADLINE: {c['headline']}\nTEXT: {c['text'][:600]}"
        for i, c in enumerate(candidates, 1)
    )

    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {'role': 'system', 'content': RERANK_PROMPT.format(top_k=top_k)},
                {'role': 'user', 'content': f'Question:\n{question}\n\nCandidates:\n{packed}'},
            ],
            temperature=0.0,   # deterministic ranking
            max_tokens=1000
        )

        # Extract the JSON-like list, e.g. [3, 1, 7]
        raw = resp.choices[0].message.content or ''
        m = re.search(r'\[[\d,\s]*\]', raw)
        indices = [int(n) for n in re.findall(r'\d+', m.group(0))] if m else []
    except Exception as e:
        print(f'rerank_contexts failed: {e}')
        return candidates[:top_k]

    # Validate indices, remove duplicates and keep the LLM order
    seen, result = set(), []
    for i in indices:
        if 1 <= i <= len(candidates) and i not in seen:
            seen.add(i)
            result.append(candidates[i - 1])
    return result[:top_k] or candidates[:top_k]

---
## Answer Generation

The final answer is generated only from the reranked contexts. The system prompt forbids invented facts
and requires the answer in the language of the user's question. Only the last few question/answer pairs
are sent as history, which keeps the token cost almost constant. Contexts are never stored in the history.

In [6]:
SYSTEM_PROMPT = '''
You answer questions about the user's projects.

Rules:
- ALWAYS answer in the same language as the user's original question
- Answer strictly from the provided context. Never invent metrics, causes or conclusions
- If the context does not contain the answer, say so in one short sentence
- If only part of the question is covered, answer that part and state what is not stated
- Use project and source information only to understand the context
- Use the earlier conversation only to understand follow-up questions; the facts must come from the provided context
- Do NOT mention chunk numbers, project names, file names or sources in the answer. Write only the content (text, numbers)
- Do NOT ask follow-up questions or add closing sentences

Output:
- Start with a short plain-language summary for a non-expert
- If the user explicitly requests an expert, manager, scoring, or detailed review, give a professional interpretation with evidence from the context
- Keep the answer practical, specific, and grounded in evidence
- Use short bullets when useful
'''

def generate_answer(question, contexts, conversation=None):
    if not contexts:
        return 'No relevant context was found.'

    # Numbered context block built from the reranked chunks
    context_block = '\n\n'.join(
        f"[{i}] SOURCE: {c['source']}\nHEADLINE: {c['headline']}\nTEXT: {c['text']}"
        for i, c in enumerate(contexts, 1)
    )

    # Messages: system prompt -> recent history (questions and answers only) -> current question with context
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for t in (conversation or [])[-(HISTORY_MESSAGES // 2):]:
        messages.append({'role': 'user', 'content': t['question']})
        messages.append({'role': 'assistant', 'content': t['answer']})
    messages.append({'role': 'user', 'content': f'Question:\n{question}\n\nContext:\n{context_block}'})

    try:
        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            temperature=0.3,   # low variation, factual answers with slightly natural wording
            max_tokens=3000
        )
        return (resp.choices[0].message.content or '').strip()
    except Exception as e:
        return f'generate_answer failed: {e}'

---
## Main Chat Function

`chat()` runs the full pipeline
- rewrite, retrieve, rerank, generate
- each question and answer is stored in `conversation`, which serves both as the memory for follow-up questions and as the source for the export
- use `verbose=True` to inspect the queries and selected contexts

`reset_chat()` to start a new topic

In [7]:
# Conversation memory: [{'timestamp', 'question', 'answer'}]
conversation = []

def chat(question, verbose=False):
    queries = rewrite_query(question, conversation)
    candidates = embedding_retrieval(queries)
    contexts = rerank_contexts(' | '.join(queries), candidates)

    # Optional debug output
    if verbose:
        print('Queries:')
        for q in queries:
            print(f'  - {q}')
        print(f'Candidates: {len(candidates)}, contexts: {len(contexts)}')
        for i, c in enumerate(contexts, 1):
            print(f"  [{i}] {c['source']} | {c['headline']} | score={c['score']:.4f}")
        print('\nAnswer:')

    answer = generate_answer(question, contexts, conversation)
    print(answer)

    conversation.append({
        'timestamp': datetime.now().strftime('%d.%m.%Y %H:%M:%S'),
        'question': question,
        'answer': answer,
    })


def reset_chat():
    conversation.clear()

---
## Export Conversation

In [8]:
def export_conversation():
    base_name = results_tab / f'conversation_{datetime.now():%d.%m.%Y_%H-%M}'

    # JSON: full structured history
    Path(f'{base_name}.json').write_text(
        json.dumps(conversation, ensure_ascii=False, indent=2), encoding='utf-8')

    # TXT: readable blocks (time, question, answer) separated by an empty line
    Path(f'{base_name}.txt').write_text(
        f"\n{'-'*40}\n\n".join(
            f'[{t['timestamp']}]\nQ: {t['question']}\n\nA: {t['answer']}'
            for t in conversation),
        encoding='utf-8'
    )

    print(f'Conversation saved:\n{base_name}.json,\n{base_name}.txt')

---
## Example Usage

The questions below are intentionally in Slovak to show that the engine answers in the language of the question,
even though the knowledge base is in English.

In [9]:
# Standalone question
chat('Ako sa vypočítal lift?')

Lift sa počítal ako pomer priemeru v danej skupine voči celkovému priemeru.

- **Rizikové skupiny (buckets):** v každej z 200 iterácií sa z každej skupiny náhodne vybralo 50 záznamov. Lift = priemerný „evidence" (dôkazový signál) v skupine / globálny priemer. Výsledné hodnoty sa zobrazili v boxplote (lift_distribution.png).
- **Doménový profil (shipping/discount):** pre štyri modely sa vzal top 1 % záznamov. V nich sa spočítal priemerný surový signál pre dopravu (shipping) a zľavu (discount). Lift je tento priemer voči globálnemu priemeru.

Kontext neuvádza presný vzorec ani to, ako je definovaný „evidence" signál nad rámec uvedeného.


In [10]:
# Follow-up question: resolved through conversation history
chat('Spomenul si priemerný signál dopravy (shipping) a zľavy (discount). Ako sa vypočítali tieto ukazovatele?')

Stručne: pre každý z modelov sa vzal top 1 % najpodozrivejších záznamov a v nich sa spočítal priemer surových signálov dopravy a zľavy. Lift je potom tento priemer voči globálnemu priemeru.

**Domain profile (porovnanie modelov)**
- Pre štyri modely sa znovu vytvorili masky top 1 %.
- V top 1 % sa vypočítal priemer surového signálu dopravy (`shipping_anomaly_raw`) a zľavy (`discount_anomaly_raw`).
- Lift = priemer v top 1 % voči globálnemu priemeru. Výsledky sa uložili do `model_domain_profile.csv`.

**Surový signál dopravy (`shipping_anomaly_raw`)**
- Najprv sa spočíta `shipping_delay` ako skutočné mínus plánované trvanie dodania.
- K dátam sa pripoja štatistiky podľa spôsobu dopravy (Shipping Mode) a trhu (Market).
- `shipping_deviation` je odchýlka od mediánu danej skupiny.
- `shipping_anomaly_raw` = absolútna odchýlka / (IQR + 1e-5), teda robustné škálovanie.
- Po feature engineeringu je hodnota zhora ohraničená približne na 1,0.

**Surový signál zľavy (`discount_anomaly_raw`)**
- 

In [11]:
# New topic
chat('Povedz mi základné informácie o market-event-impact')

Ide o projekt, ktorý analyzuje vplyv 21 makroekonomických udalostí na index DAX a menový pár EUR/USD v troch krízových obdobiach (2020–2026).

**Dáta a príprava**
- Použité sú minútové (1-minútové) dáta DAX a EUR/USD a kalendár makroekonomických oznámení.
- Z približne 89 000 oznámení sa podľa výberových kritérií vybralo 21 finálnych udalostí.
- Okolo každej udalosti sa vytvárajú okná s OHLC dátami v časových rámcoch M1 až M60.
- Datasety majú spolu približne 477 MB a sú uložené na Hugging Face.

**Metódy**
- Porovnávajú sa modely GARCH a GJR-GARCH (AIC, BIC, perzistencia, konvergencia).
- GJR-GARCH vychádza lepšie v ~19 % okien udalostí, podobne v ~73 % a GARCH lepšie v ~8 %.

**Hlavné zistenia**
- Najväčšie výkyvy spôsobujú rozhodnutia ECB o sadzbách a americký NFP.
- DAX reaguje pomalšie ako EUR/USD.
- Volatilita závisí od režimu: rozlišujú sa pokojné a stresové obdobia s dlhými chvostmi rozdelenia.
- Jednotný prístup k riadeniu rizika preto nefunguje, vhodnejšie je dynamické riaden

In [12]:
# Out-of-scope question: the engine should say the answer is not in the context
chat('Koľko stojí čaj v obchode?')

Kontext neobsahuje informáciu o cene čaju v obchode.


In [13]:
# Out-of-scope question, then inspect the pipeline with verbose=True
chat('Koľko stojí čaj v obchode?', verbose=True)

Queries:
  - tea price in store
  - cost of tea in a shop
  - tea price retail dataset
Candidates: 20, contexts: 1
  [1] notebooks\01_data_preprocessing_and_eda.ipynb | Data type optimization section and statistics after cleaning | score=0.1573

Answer:
Kontext neobsahuje informáciu o cene čaju v obchode.


In [14]:
# Save the whole conversation to JSON and TXT
export_conversation()

Conversation saved:
A:\Vzdelávanie\GitHub_portfolio\project-miner-assistant\results\tables\conversation_13.10.2026_17-16.json,
A:\Vzdelávanie\GitHub_portfolio\project-miner-assistant\results\tables\conversation_13.10.2026_17-16.txt


---
## Key Takeaways

- Built a complete RAG pipeline on top of the **ChromaDB knowledge base**: query rewriting, vector retrieval, LLM reranking and grounded answer generation
- **Query rewriting** turns questions in any language into self-contained english search queries, splits multi-part questions and resolves follow-ups
- **Retrieval** embeds all queries in one API call and one ChromaDB request, merges results by rank and removes duplicates; the **reranker** keeps the 6 most useful contexts
- **Answers** use only the retrieved context, are written in the language of the question and openly state when information is missing
- **Conversation memory** is limited to the last 3 question/answer pairs and never stores contexts, so the token cost per answer stays almost constant
- Every LLM step has a **fallback** (original question or top vector candidates), so one failed call does not stop the pipeline
- The conversation can be exported to **JSON and TXT**
- Same functions can later be reused in a Streamlit app